This is the notebook to run inference on trained model using the `CoppeliaSim` robot simulator. Follow the below steps for running this notebook

1. Download and install CoppeliaSim from here: [https://www.coppeliarobotics.com/](https://www.coppeliarobotics.com/)
2. Open the scene file `precision_spraying\sim_envs\uav_common_env.ttt` in the simulator software (must do it before running the notebook).
3. Run all the cells in the notebook one by one.
4. For every environment, modify the path for trained model and run the cell.
4. **IMPORTANT**: When closing the CoppeliaSim scene file, please select "No" for the question "Do you wish to save the changes?"

#### Load modules and configurations

In [1]:
# ================================
# Hyperparameters and paths
# ================================
exp_alg = 'crossq'
exp_set = 'set1'
sf = 10                         # scaling factor
num_robots = 2
max_steps = 1000
json_path = rf'..\exp_sets\cont_sets.json'

# ================================
# Imports
# ================================
import numpy as np
import json
import copy
import itertools
import pygame
import gymnasium as gym
np.random.seed(33)  # seeding

from coppeliasim_zmqremoteapi_client import RemoteAPIClient
if exp_alg == 'crossq':
    from sb3_contrib import CrossQ as model_loader

# ================================
# Utility Functions  (unchanged — model depends on these)
# ================================
def binary_list_to_decimal(bin_list):
    bin = ''
    for b in bin_list:
        bin += str(b)
    dec = int(bin, 2)
    return dec

def is_inside_polygon(point, poly):
    x, y = point
    inside = False
    n = len(poly)
    p1x, p1y = poly[0]
    for i in range(n + 1):
        p2x, p2y = poly[i % n]
        if min(p1y, p2y) < y <= max(p1y, p2y) and x <= max(p1x, p2x):
            if p1y != p2y:
                xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
            if p1x == p2x or x <= xinters:
                inside = not inside
        p1x, p1y = p2x, p2y
    return inside

def min_dist(x):
    x = np.array(x).astype('float32')
    dists = []
    for p1, p2 in itertools.combinations(x, 2):
        dist = np.linalg.norm(p1 - p2)
        dists.append(dist)
    return float(np.min(dists))

def load_experiment_dict_json(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)
    for set_name, cfg in data.items():
        cfg["field"] = [tuple(p) for p in cfg["field"]]
        cfg["init_positions"] = [np.array(p, dtype=float) for p in cfg["init_positions"]]
        cfg["infected_locations"] = [tuple(p) for p in cfg["infected_locations"]]
    return data

def load_experiment(exp_set, json_path=json_path, scaling_factor=sf):
    """Load and scale a single experiment set by name (e.g. 'set1', 'set2')."""
    raw = load_experiment_dict_json(json_path)
    assert exp_set in raw, f"'{exp_set}' not found in {json_path}. Available: {list(raw.keys())}"
    cfg = copy.deepcopy(raw[exp_set])
    cfg['field']              = [(x * scaling_factor, y * scaling_factor) for (x, y) in cfg['field']]
    cfg['infected_locations'] = [(x * scaling_factor, y * scaling_factor) for (x, y) in cfg['infected_locations']]
    cfg['init_positions']     = [v * scaling_factor for v in cfg['init_positions']]
    return cfg

selected_experiment = load_experiment(exp_set)  # default set used by the environment's default arg

# ================================
# Environment  (unchanged — trained model depends on this exactly)
# ================================
class MultiRobotEnv(gym.Env):
    metadata = {'render_modes': ['human', 'print', 'rgb_array'], "render_fps": 4}

    def __init__(self, render_mode=None, field_info=copy.deepcopy(selected_experiment), wind_par=[0, 0], num_robots=3):
        super(MultiRobotEnv, self).__init__()
        self.edge_buffer = 10
        self.poly_vertices = field_info['field']
        self.xs, self.ys = zip(*field_info['field'])
        self.WIDTH, self.HEIGHT = 1000, 1000

        self.num_robots = num_robots
        self.init_robot_positions = np.array(field_info['init_positions'])[:self.num_robots]
        self.robot_size = 10
        self.mass = 1.0
        self.thrust_power = 0.5
        self.max_speed = 5
        self.min_speed = -5
        self.min_positions = np.zeros(self.num_robots * 2)
        self.max_positions = np.array([[self.WIDTH, self.HEIGHT] for _ in range(self.num_robots)])
        self.min_velocities = np.array([[self.min_speed, self.min_speed] for _ in range(self.num_robots)])
        self.max_velocities = np.array([[self.max_speed, self.max_speed] for _ in range(self.num_robots)])
        self.wind_f_a, self.wind_beta_a = wind_par

        self.initial_inf_locations = field_info['infected_locations']
        self.infected_size = 10
        self.infected_length = len(field_info['infected_locations'])
        self.infected_state_length = 2 ** (self.infected_length)

        self.action_space = gym.spaces.Box(low=-1.0, high=1.0, shape=(self.num_robots, 2), dtype=np.float32)
        self.observation_space = gym.spaces.Box(
            low=np.concatenate((self.min_positions.flatten(), self.min_velocities.flatten(), np.array([0]))),
            high=np.concatenate((self.max_positions.flatten(), self.max_velocities.flatten(), np.array([self.infected_state_length - 1]))),
            dtype=np.float32)

        assert render_mode is None or render_mode in self.metadata["render_modes"]
        self.render_mode = render_mode
        self.screen = None
        self.clock = None

        self.reset()

    def _get_obs(self):
        info = {f'robot{i}': self.robot_positions[i] for i in range(self.num_robots)}
        infected = binary_list_to_decimal(list(self.infected_dict.values()))
        state = np.concatenate((self.robot_positions.flatten(), self.robot_velocities.flatten(), np.array([infected])), dtype=np.float32)
        return state, info

    def reset(self, seed=None, options={}):
        self.step_count = 0
        self.visited = set()
        self.infected_locations = copy.deepcopy(self.initial_inf_locations)
        self.infected_dict = {v: 0 for v in self.infected_locations}
        self.robot_positions = copy.deepcopy(self.init_robot_positions)
        self.robot_velocities = np.zeros((self.num_robots, 2))
        return self._get_obs()

    def step(self, actions):
        terminated, truncated = False, False
        rewards = 0
        self.step_count += 1
        for i in range(self.num_robots):
            ax, ay = actions[i] * self.thrust_power

            self.robot_velocities[i][0] += ax / self.mass + self.wind_f_a * np.cos(np.radians(self.wind_beta_a))
            self.robot_velocities[i][1] += ay / self.mass + self.wind_f_a * np.sin(np.radians(self.wind_beta_a))
            self.robot_velocities[i] = np.clip(self.robot_velocities[i], self.min_speed, self.max_speed)

            new_position = self.robot_positions[i] + self.robot_velocities[i]
            if is_inside_polygon(new_position, self.poly_vertices):
                pass
            else:
                rewards -= 10000
                self.robot_velocities[i][:] = 0

            self.robot_positions[i] += self.robot_velocities[i]
            self.robot_positions[i] = np.clip(self.robot_positions[i], [0, 0], [self.WIDTH, self.HEIGHT])

            if tuple(self.robot_positions[i]) in self.visited:
                rewards -= 100
            else:
                rewards -= 10
            self.visited.add(tuple(self.robot_positions[i]))

            nearby_infected_locations = []
            for j, inf_loc in enumerate(self.infected_locations):
                dist = np.linalg.norm(self.robot_positions[i] - inf_loc)
                if dist <= self.infected_size:
                    nearby_infected_locations.append(inf_loc)
                    rewards += 10000
            for inf_loc in nearby_infected_locations:
                self.infected_locations.remove(inf_loc)
                self.infected_dict[tuple(inf_loc)] = 1

        if len(self.infected_locations) == 0:
            rewards += 100000
            terminated = True

        if self.num_robots > 1:
            min_dist_between_robots = min_dist(self.robot_positions)
            if min_dist_between_robots < self.robot_size:
                rewards = -100000
                terminated = True

        obs, info = self._get_obs()
        return obs, rewards, terminated, truncated, info

    def render(self):
        if self.screen is None and self.render_mode == "human":
            pygame.init()
            pygame.display.init()
            self.screen = pygame.display.set_mode((self.WIDTH, self.HEIGHT))
            pygame.display.set_caption("Multi-robot RL Environment")
            if self.clock is None:
                self.clock = pygame.time.Clock()
                self.running = True

        self.screen.fill((255, 255, 255))
        colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 128, 0), (128, 0, 255), (255, 0, 255), (128, 128, 128)]
        pix_size = 10

        pygame.draw.polygon(surface=self.screen, color=(255, 255, 0), points=self.poly_vertices)

        for point in self.visited:
            pygame.draw.circle(self.screen, pygame.Color(100, 100, 100, a=0.2), point, pix_size / 2)

        for i in range(self.num_robots):
            pygame.draw.circle(self.screen, colors[i], (int(self.robot_positions[i][0]), int(self.robot_positions[i][1])), pix_size / 2)
            for l in self.infected_locations:
                pygame.draw.circle(self.screen, (0, 255, 255), (int(l[0]), int(l[1])), pix_size / 2)

        pygame.display.flip()
        self.clock.tick(60)

    def close(self):
        if self.screen is not None:
            pygame.display.quit()
            pygame.quit()

# ================================
# Register environment
# ================================
gym.register(id='MultiRobotEnv-v0',
             entry_point=MultiRobotEnv,
             max_episode_steps=max_steps)

# ================================
# Initialize the ZeroMQ Client
# ================================
client = RemoteAPIClient()
sim = client.getObject('sim')
defaultIdleFps = sim.getInt32Param(sim.intparam_idle_fps)
sim.setInt32Param(sim.intparam_idle_fps, 0)

# ================================
# Drone Simulator  (updated to match Code 2 structure)
# ================================
class Drone_simulator:
    def __init__(self, gym_env, scaling_factor=50, height=0.35, num_robots=2):
        self.scaling_factor = scaling_factor
        self.polygon = gym_env.unwrapped.poly_vertices  # raw world-space polygon
        self.scaled_polygon = [(x / scaling_factor, y / scaling_factor) for (x, y) in self.polygon]
        self.rounded_polygon = self.scaled_polygon + [self.scaled_polygon[0]]
        self.weed_locations = list(gym_env.unwrapped.initial_inf_locations)
        self.height = height
        self.num_robots = num_robots

        # Drawing handles
        self.field_drawing = None   # set by draw_field()
        self.trace_line = None      # set by start_simulation()

        # Track spawned weeds so they can be cleaned up on stop
        self.spawned_weeds = []

        # Cache quadcopters and their initial positions
        self.all_drones = []
        self.initial_positions = {}
        i = 0
        while True:
            h = sim.getObject(f"/Quadcopter[{i}]", {'noError': True})
            if h == -1:
                break
            self.all_drones.append(h)
            self.initial_positions[h] = sim.getObjectPosition(h, -1)
            i += 1

    # ── Simulation lifecycle ────────────────────────────────────────

    def start_simulation(self):
        # Trace line for robot paths (one shared drawing object)
        self.trace_line = sim.addDrawingObject(
            sim.drawing_lines, 5, 0, -1, 99999, [255, 0, 0])
        sim.startSimulation()
        print("Simulation started")

    def stop_simulation(self):
        sim.stopSimulation()
        # Wait until the simulator is fully stopped before cleaning up
        while sim.getSimulationState() != sim.simulation_stopped:
            sim.step()

        # Remove trace line
        if self.trace_line is not None:
            try:
                sim.removeDrawingObject(self.trace_line)
            except Exception as e:
                print(f"Warning: could not remove trace line: {e}")
            self.trace_line = None

        # Remove field drawing
        self.clear_field()

        # Remove spawned weeds
        self.clear_weeds()

        # Restore all drones to their initial positions
        for drone, pos in self.initial_positions.items():
            sim.setObjectPosition(drone, -1, pos)
            sim.setModelProperty(drone, 0)

        print("Simulation stopped and robots reset")

    # ── Field drawing ───────────────────────────────────────────────

    def draw_field(self):
        self.field_drawing = sim.addDrawingObject(
            sim.drawing_lines, 5, 0, -1, 9999, [255, 255, 255])
        for i in range(len(self.rounded_polygon) - 1):
            p1 = self.rounded_polygon[i]
            p2 = self.rounded_polygon[i + 1]
            line = [p1[0], p1[1], 0.1,
                    p2[0], p2[1], 0.1]
            sim.addDrawingObjectItem(self.field_drawing, line)

    def clear_field(self):
        if self.field_drawing is not None:
            try:
                sim.removeDrawingObject(self.field_drawing)
            except Exception as e:
                print(f"Warning: could not remove field drawing: {e}")
            self.field_drawing = None

    # ── Weed management ─────────────────────────────────────────────

    def set_weed_locations(self):
        weed_template = sim.getObject('/weed')
        self.clear_weeds()  # Remove any previously spawned weeds first
        for loc in self.weed_locations:
            x = [xi / self.scaling_factor for xi in loc]
            new_pos = x + [0]
            new_weed = sim.copyPasteObjects([weed_template])[0]
            sim.setObjectPosition(new_weed, -1, new_pos)
            self.spawned_weeds.append(new_weed)

    def clear_weeds(self):
        for obj in self.spawned_weeds:
            if sim.isHandle(obj):
                sim.removeObject(obj)
        self.spawned_weeds = []

    # ── Agent control ───────────────────────────────────────────────

    def set_agent_positions(self, info):
        """Place drones at initial positions; hide drones not in use."""
        for i, drone in enumerate(self.all_drones):
            if i < self.num_robots:
                pos = info[f'robot{i}']  # Code-1 info: direct position array
                pos = [p / self.scaling_factor for p in pos] + [self.height]
                sim.setObjectPosition(drone, -1, pos)
                sim.setObjectInt32Param(drone, sim.objintparam_visibility_layer, 1)
            else:
                # Hide drones that are not part of this experiment
                sim.setModelProperty(
                    drone,
                    sim.modelproperty_not_visible
                    | sim.modelproperty_not_collidable
                    | sim.modelproperty_not_detectable
                    | sim.modelproperty_not_dynamic
                )

    def move_agents(self, info):
        """Move drones to current robot positions and draw trace lines."""
        for i in range(self.num_robots):
            target = sim.getObject(f"/target[{i}]")
            prev_pos = sim.getObjectPosition(target, -1)

            pos = info[f'robot{i}']  # Code-1 info: direct position array
            pos = [p / self.scaling_factor for p in pos] + [self.height]
            sim.setObjectPosition(target, -1, pos)

            # Draw trace line between previous and current target position
            if self.trace_line is not None:
                line_data = prev_pos + pos
                sim.addDrawingObjectItem(self.trace_line, line_data)

# ================================
# run_simulation()
# ================================
def run_simulation(trained_model_path, exp_set=exp_set, num_robots=num_robots,
                   scaling_factor=50, height=0.35):
    """
    Run a simulation episode.

    Parameters
    ----------
    trained_model_path : str
        Path to the saved model (.zip).
    exp_set : str
        Experiment set name, e.g. 'set1', 'set2', ...
        Defaults to the module-level ``exp_set`` variable.
    num_robots : int
        Number of active robots. Must be <= init positions available in the set.
        Defaults to the module-level ``num_robots`` variable.
    scaling_factor : int
        CoppeliaSim world scaling factor.
    height : float
        Flight height in CoppeliaSim (metres).
    """
    # Load trained model
    model = model_loader.load(trained_model_path)

    # Load and scale the requested experiment set
    field_info = load_experiment(exp_set)
    assert num_robots <= len(field_info['init_positions']), (
        f"num_robots={num_robots} exceeds the {len(field_info['init_positions'])} "
        f"init positions available in '{exp_set}'."
    )
    print(f"Running: exp_set='{exp_set}', num_robots={num_robots}")

    # Create Gym environment — field_info and num_robots are passed explicitly
    env = gym.make(
        'MultiRobotEnv-v0',
        render_mode='human',
        field_info=field_info,
        num_robots=num_robots
    )
    env.metadata['render_fps'] = 1
    obs, info = env.reset()
    env.render()

    # Set up CoppeliaSim drone simulator
    drone_simulator = Drone_simulator(
        gym_env=env,
        scaling_factor=scaling_factor,
        height=height,
        num_robots=num_robots
    )
    drone_simulator.draw_field()
    drone_simulator.set_agent_positions(info=info)
    drone_simulator.set_weed_locations()
    print("Initial info:", info)

    input("Press Enter to start simulation...")

    # Start simulation
    drone_simulator.start_simulation()
    terminated, truncated = False, False
    total_rewards = 0

    while True:
        action, _ = model.predict(obs)
        obs, reward, terminated, truncated, info = env.step(list(action))
        env.render()
        total_rewards += reward
        print(
            f"Obs: {obs}, "
            f"Reward: {reward:.2f}, "
            f"Total: {total_rewards:.2f}, "
            f"terminated: {terminated}"
        )
        drone_simulator.move_agents(info=info)

        if terminated or truncated:
            print(f"Episode finished — terminated: {terminated}, truncated: {truncated}")
            break
        pygame.event.get()
    return env, drone_simulator

pygame-ce 2.5.7 (SDL 2.32.10, Python 3.14.4)


#### Run simulation on two environment variations

#### Environment 1 with two UAVs

In [2]:
# ================================
# Entry point
# ================================
trained_model_path = rf"..\trained_models\cont_env1_2robots_CrossQ"
env, drone_simulator = run_simulation(
    trained_model_path=trained_model_path,
    exp_set='set1',     # change to 'set2', 'set3', etc.
    num_robots=2        # change to any valid number for the chosen set
)

Running: exp_set='set1', num_robots=2


c:\Users\mabari\.conda\envs\ai_workshop\Lib\site-packages\gymnasium\spaces\box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(


Initial info: {'robot0': array([140., 340.]), 'robot1': array([400., 250.])}
Simulation started
Obs: [ 1.4048697e+02  3.4041629e+02  3.9951025e+02  2.4977576e+02
  4.8697442e-01  4.1629595e-01 -4.8974204e-01 -2.2424152e-01
  0.0000000e+00], Reward: -20.00, Total: -20.00, terminated: False
Obs: [ 1.4146373e+02  3.4114395e+02  3.9862018e+02  2.4970683e+02
  9.7675806e-01  7.2766358e-01 -8.9007699e-01 -6.8930537e-02
  0.0000000e+00], Reward: -20.00, Total: -40.00, terminated: False
Obs: [ 1.4293289e+02  3.4221561e+02  3.9730740e+02  2.4935075e+02
  1.4691570e+00  1.0716498e+00 -1.3127780e+00 -3.5608113e-01
  0.0000000e+00], Reward: -20.00, Total: -60.00, terminated: False
Obs: [144.88649    343.56256    395.5247     248.8995       1.9535973
   1.3469635   -1.7827127   -0.45123544   0.        ], Reward: -20.00, Total: -80.00, terminated: False
Obs: [147.32541   344.87247   393.3466    248.29288     2.4389153   1.3098795
  -2.1781096  -0.6066272   0.       ], Reward: -20.00, Total: -100.00,

In [3]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

Simulation stopped and robots reset


#### Environment 3 with 3 UAVs

In [4]:
# ================================
# Entry point
# ================================
trained_model_path = rf"..\trained_models\cont_env3_3robots_CrossQ.zip"
env, drone_simulator = run_simulation(
    trained_model_path=trained_model_path,
    exp_set='set3',     # change to 'set2', 'set3', etc.
    num_robots=3        # change to any valid number for the chosen set
)

Running: exp_set='set3', num_robots=3
Initial info: {'robot0': array([200., 250.]), 'robot1': array([300., 300.]), 'robot2': array([250., 300.])}
Simulation started
Obs: [ 2.0048398e+02  2.4951889e+02  2.9976263e+02  2.9981589e+02
  2.4951387e+02  3.0041962e+02  4.8397422e-01 -4.8110902e-01
 -2.3737869e-01 -1.8412113e-01 -4.8612735e-01  4.1961038e-01
  4.0000000e+01], Reward: 19970.00, Total: 19970.00, terminated: False
Obs: [ 2.0142374e+02  2.4854521e+02  2.9905750e+02  2.9919809e+02
  2.4852800e+02  3.0037662e+02  9.3975687e-01 -9.7367430e-01
 -7.0512003e-01 -6.1779535e-01 -9.8587561e-01 -4.2999744e-02
  4.0000000e+01], Reward: -30.00, Total: 19940.00, terminated: False
Obs: [202.84442    247.427      297.99826    298.88242    247.04251
 299.85376      1.4206952   -1.1182129   -1.0592434   -0.31567127
  -1.4854884   -0.52284753  40.        ], Reward: -30.00, Total: 19910.00, terminated: False
Obs: [ 2.0475963e+02  2.4634911e+02  2.9652939e+02  2.9867374e+02
  2.4505760e+02  2.9885394

In [5]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

Simulation stopped and robots reset
